<div dir="rtl">

# بررسی مستقل اتصال API

این نوت‌بوک از داده‌ها و سل‌های پروژه مستقل است. کلید و آدرس خصوصی چاپ یا داخل فایل ذخیره نمی‌شوند.

</div>


In [1]:
# API-DIAG-C01 — Configuration inspection, no network
import os
import json
import hashlib
import time
import urllib.request
import urllib.error
import urllib.parse

diag_url = os.environ.get("ARVAN_CHAT_URL", "").strip()
diag_model = os.environ.get("ARVAN_CHAT_MODEL", "").strip()
diag_keys = {
    os.environ.get(name, "").strip()
    for name in ("ARVAN_CHAT_API_KEY", "ARVAN_AI_API_KEY")
} - {""}

diag_parts = urllib.parse.urlsplit(diag_url)
diag_config_valid = (
    diag_parts.scheme == "https"
    and bool(diag_parts.hostname)
    and not diag_parts.username
    and not diag_parts.password
    and not diag_parts.query
    and not diag_parts.fragment
    and bool(diag_model)
    and len(diag_keys) == 1
)

print(json.dumps({
    "status": "CONFIG_READY" if diag_config_valid else "CONFIG_MISSING_OR_INVALID",
    "url_present": bool(diag_url),
    "model": diag_model or None,
    "distinct_nonempty_keys": len(diag_keys),
    "https": diag_parts.scheme == "https",
    "api_calls": 0,
    "secret_values_printed": False,
}, ensure_ascii=False, indent=2))


{
  "status": "CONFIG_READY",
  "url_present": true,
  "model": "Qwen3-30B-A3B",
  "distinct_nonempty_keys": 1,
  "https": true,
  "api_calls": 0,
  "secret_values_printed": false
}


<div dir="rtl">

ابتدا سل تنظیمات را اجرا کنید. اگر CONFIG_READY بود، در سل بعد RUN_DIRECT_TEST را True کنید و فقط یک بار اجرا کنید. پس از تست آن را به False برگردانید. اجرای مجدد با True یک درخواست تازه می‌فرستد.

</div>


In [2]:
# API-DIAG-C02 — One direct HTTP chat request, no SDK, no retries
RUN_DIRECT_TEST = True

if not RUN_DIRECT_TEST:
    print(json.dumps({"status": "DIRECT_TEST_DISABLED", "api_calls": 0}))
else:
    if not diag_config_valid:
        raise ValueError("Complete the environment configuration first.")

    # ARVAN_CHAT_URL is the same base URL used by the OpenAI client.
    diag_endpoint = diag_url.rstrip("/") + "/chat/completions"
    diag_body = json.dumps({
        "model": diag_model,
        "messages": [
            {"role": "system", "content": 'Reply only with {"connection":"ok"}.'},
            {"role": "user", "content": "Connection check."},
        ],
        "max_tokens": 1024,
    }, ensure_ascii=False).encode("utf-8")

    diag_request = urllib.request.Request(
        diag_endpoint,
        data=diag_body,
        headers={
            "Authorization": "Bearer " + next(iter(diag_keys)),
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )

    # Prevent redirects from forwarding credentials to a different endpoint.
    class DiagNoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    diag_opener = urllib.request.build_opener(DiagNoRedirect())
    diag_report = {
        "status": "DIRECT_HTTP_TEST_FAILED",
        "api_request_attempts": 1,
        "automatic_retries": 0,
        "http_status": None,
        "error_type": None,
        "diagnostic_headers": {},
        "expected_json_received": False,
        "response_model": None,
        "usage": None,
        "project_data_used": False,
    }
    diag_started = time.monotonic()

    try:
        with diag_opener.open(diag_request, timeout=45) as response:
            diag_raw = response.read()
            diag_report["http_status"] = response.status
            diag_report["diagnostic_headers"] = {
                name: response.headers[name]
                for name in ("x-request-id", "date", "server")
                if name in response.headers
            }

        diag_payload = json.loads(diag_raw)
        diag_choice = (diag_payload.get("choices") or [{}])[0]
        diag_content = (diag_choice.get("message") or {}).get("content")
        diag_report["response_model"] = diag_payload.get("model")
        diag_report["usage"] = diag_payload.get("usage")
        diag_report["finish_reason"] = diag_choice.get("finish_reason")

        try:
            diag_report["expected_json_received"] = (
                json.loads(diag_content) == {"connection": "ok"}
            )
        except (TypeError, ValueError):
            pass

        diag_report["model_name_tail_matches"] = (
            str(diag_payload.get("model") or "").split("/")[-1].casefold()
            == diag_model.split("/")[-1].casefold()
        )
        if (
            diag_report["http_status"] == 200
            and diag_report["expected_json_received"]
            and diag_report["model_name_tail_matches"]
            and diag_report["finish_reason"] == "stop"
        ):
            diag_report["status"] = "DIRECT_HTTP_SMALL_CHAT_PASS"
        else:
            diag_report["status"] = "DIRECT_HTTP_RESPONSE_REQUIRES_INSPECTION"

    except urllib.error.HTTPError as error:
        diag_error_body = error.read()
        diag_report.update({
            "http_status": error.code,
            "error_type": type(error).__name__,
            "error_body_bytes": len(diag_error_body),
            "error_body_sha256": hashlib.sha256(diag_error_body).hexdigest(),
            "diagnostic_headers": {
                name: error.headers[name]
                for name in ("x-request-id", "date", "server")
                if name in error.headers
            },
        })
    except Exception as error:
        diag_report["error_type"] = type(error).__name__

    diag_report["elapsed_seconds"] = round(
        time.monotonic() - diag_started, 2
    )
    print(json.dumps(diag_report, ensure_ascii=False, indent=2))


{
  "status": "DIRECT_HTTP_TEST_FAILED",
  "api_request_attempts": 1,
  "automatic_retries": 0,
  "http_status": 502,
  "error_type": "HTTPError",
  "diagnostic_headers": {
    "x-request-id": "fde9c5c239832eb1e0ef19d15f9e6724",
    "date": "Tue, 06 Oct 2026 21:53:22 GMT",
    "server": "ArvanCloud"
  },
  "expected_json_received": false,
  "response_model": null,
  "usage": null,
  "project_data_used": false,
  "error_body_bytes": 140,
  "error_body_sha256": "9dc38cea5772e62fb54a88132a65af6fd475bd22addb43d6e5d62ceb38959c4d",
  "elapsed_seconds": 7.61
}


In [3]:
# API-DIAG-C03 — Inspect current configuration and SDK routing without network
import importlib.metadata
import httpx
from openai import OpenAI

api03_url = os.environ.get("ARVAN_CHAT_URL", "").strip()
api03_model = os.environ.get("ARVAN_CHAT_MODEL", "").strip()
api03_keys = {
    os.environ.get(name, "").strip()
    for name in ("ARVAN_CHAT_API_KEY", "ARVAN_AI_API_KEY")
} - {""}

api03_report = {
    "api_calls": 0,
    "configured_model": api03_model or None,
    "distinct_nonempty_keys": len(api03_keys),
    "c01_url_matches_current_environment": diag_url == api03_url,
    "c01_model_matches_current_environment": diag_model == api03_model,
    "c01_keys_match_current_environment": diag_keys == api03_keys,
    "openai_version": importlib.metadata.version("openai"),
    "httpx_version": importlib.metadata.version("httpx"),
}

api03_parts = urllib.parse.urlsplit(api03_url)
api03_valid = (
    api03_parts.scheme == "https"
    and bool(api03_parts.hostname)
    and not api03_parts.username
    and not api03_parts.password
    and not api03_parts.query
    and not api03_parts.fragment
    and bool(api03_model)
    and len(api03_keys) == 1
)

if api03_valid:
    api03_captured = []

    def api03_mock(request):
        api03_captured.append(str(request.url))
        return httpx.Response(200, json={
            "id": "local-routing-check",
            "object": "chat.completion",
            "created": 0,
            "model": api03_model,
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": "local"},
                "finish_reason": "stop",
            }],
        })

    with OpenAI(
        api_key=next(iter(api03_keys)),
        base_url=api03_url,
        max_retries=0,
        http_client=httpx.Client(
            transport=httpx.MockTransport(api03_mock)
        ),
    ) as api03_client:
        api03_client.chat.completions.create(
            model=api03_model,
            messages=[{"role": "user", "content": "Local route check"}],
            max_tokens=1,
        )

    api03_report["sdk_route_matches_direct_route"] = (
        len(api03_captured) == 1
        and api03_captured[0]
        == api03_url.rstrip("/") + "/chat/completions"
    )
    api03_report["status"] = "LOCAL_ROUTING_CHECK_COMPLETE"
else:
    api03_report["status"] = "CONFIG_MISSING_OR_INVALID"

print(json.dumps(api03_report, ensure_ascii=False, indent=2))

{
  "api_calls": 0,
  "configured_model": "Qwen3-30B-A3B",
  "distinct_nonempty_keys": 1,
  "c01_url_matches_current_environment": true,
  "c01_model_matches_current_environment": true,
  "c01_keys_match_current_environment": true,
  "openai_version": "1.75.0",
  "httpx_version": "0.24.1",
  "sdk_route_matches_direct_route": true,
  "status": "LOCAL_ROUTING_CHECK_COMPLETE"
}


In [20]:
# API-DIAG-C04 — Gateway test using Authorization: apikey
import json
import time
import hashlib
import ssl
import urllib.request
import urllib.error
from pathlib import Path


def api04_run():
    report = {
        "status": "CONFIG_NOT_READY",
        "api_request_attempts": 0,
        "automatic_retries": 0,
        "http_status": None,
        "response_model": None,
        "expected_json_received": False,
        "usage": None,
        "error_type": None,
        "project_data_used": False,
        "authorization_scheme": "apikey",
    }

    env_path = Path(
        "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
        "RAG-projects/RAG_finance/.env"
    )

    report["env_file_found"] = env_path.is_file()
    if not report["env_file_found"]:
        report["status"] = "PROJECT_ENV_NOT_FOUND"
        return report

    # Read the file directly; do not reuse old environment variables.
    try:
        from dotenv import dotenv_values
    except ImportError:
        config = {}
        for line in env_path.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip()

            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[7:].strip()

            name, separator, value = line.partition("=")
            if not separator:
                continue

            value = value.strip()
            if value.startswith(("'", '"')):
                quote = value[0]
                end = value.find(quote, 1)
                if end < 0:
                    report["status"] = "ENV_QUOTING_INVALID"
                    return report
                value = value[1:end]
            else:
                value = value.split(" #", 1)[0].rstrip()

            config[name.strip()] = value
    else:
        config = dotenv_values(env_path, interpolate=False)

    base = (config.get("ARVAN_GATEWAY_URL") or "").strip().rstrip("/")
    key = (config.get("ARVAN_GATEWAY_API_KEY") or "").strip()
    model = (config.get("ARVAN_CHAT_MODEL") or "").strip()

    report.update(
        gateway_url_present=bool(base),
        gateway_key_present=bool(key),
        configured_model=model or None,
    )

    if not base or not key or not model:
        report["status"] = "ENV_GATEWAY_CONFIGURATION_INCOMPLETE"
        return report

    if "*" in key or key.startswith("<"):
        report["status"] = "MASKED_OR_PLACEHOLDER_KEY"
        return report

    if base != "https://ai-diag.ragfinance.ir/v1":
        report["status"] = "ENV_URL_IS_NOT_NEW_DIAGNOSTIC_GATEWAY"
        return report

    # Redact credentials and private endpoint URLs from error text.
    sensitive_values = {key}
    for name, value in config.items():
        if isinstance(value, str) and value:
            if "KEY" in name.upper() or "TOKEN" in name.upper():
                sensitive_values.add(value)
            elif "URL" in name.upper() and value != base:
                sensitive_values.add(value)

    def safe_error_text(raw):
        text = raw.decode("utf-8", errors="replace")
        for value in sorted(sensitive_values, key=len, reverse=True):
            text = text.replace(value, "[REDACTED]")
        return text[:4000]

    def diagnostic_headers(headers):
        return {
            name: headers[name]
            for name in ("x-request-id", "date", "server", "retry-after")
            if headers.get(name) is not None
        }

    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    payload = {
        "model": model,
        "messages": [
            {
                "role": "system",
                "content": 'Reply only with {"connection":"ok"}. No explanation.',
            },
            {"role": "user", "content": "Connection check."},
        ],
        "max_tokens": 1024,
        "stream": False,
    }

    request = urllib.request.Request(
        base + "/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": "apikey " + key,
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )

    # Keep TLS certificate verification enabled.
    opener = urllib.request.build_opener(
        NoRedirect(),
        urllib.request.HTTPSHandler(context=ssl.create_default_context()),
    )

    started = time.monotonic()
    report["status"] = "NEW_GATEWAY_TEST_FAILED"
    report["api_request_attempts"] = 1

    try:
        with opener.open(request, timeout=45) as response:
            raw = response.read()
            report["http_status"] = response.status
            report["diagnostic_headers"] = diagnostic_headers(response.headers)

        try:
            result = json.loads(raw)
        except (ValueError, UnicodeDecodeError):
            report["status"] = "NEW_GATEWAY_NON_JSON_RESPONSE"
            report["response_body_preview"] = safe_error_text(raw)
            return report

        if not isinstance(result, dict):
            report["status"] = "NEW_GATEWAY_UNEXPECTED_JSON_STRUCTURE"
            return report

        choices = result.get("choices")
        choice = (
            choices[0]
            if isinstance(choices, list)
            and choices
            and isinstance(choices[0], dict)
            else {}
        )
        message = choice.get("message")
        content = message.get("content") if isinstance(message, dict) else None

        report.update(
            response_model=result.get("model"),
            usage=result.get("usage"),
            finish_reason=choice.get("finish_reason"),
            nonempty_content=isinstance(content, str) and bool(content.strip()),
            model_name_tail_matches=(
                str(result.get("model") or "").split("/")[-1].casefold()
                == model.split("/")[-1].casefold()
            ),
        )

        try:
            report["expected_json_received"] = (
                json.loads(content) == {"connection": "ok"}
            )
        except (TypeError, ValueError):
            pass

        passed = (
            report["http_status"] == 200
            and report["expected_json_received"]
            and report["model_name_tail_matches"]
            and report["finish_reason"] == "stop"
        )
        report["status"] = (
            "NEW_GATEWAY_SMALL_CHAT_PASS"
            if passed
            else "NEW_GATEWAY_RESPONSE_REQUIRES_INSPECTION"
        )

        if isinstance(content, str):
            report["response_content_preview"] = safe_error_text(
                content.encode("utf-8")
            )

        if "error" in result:
            report["response_error"] = safe_error_text(
                json.dumps(result["error"], ensure_ascii=False).encode("utf-8")
            )

    except urllib.error.HTTPError as error:
        try:
            body = error.read()
            report.update(
                http_status=error.code,
                error_type=type(error).__name__,
                error_body_bytes=len(body),
                error_body_sha256=hashlib.sha256(body).hexdigest(),
                error_body_preview=safe_error_text(body),
                diagnostic_headers=diagnostic_headers(error.headers),
            )
        finally:
            error.close()

    except urllib.error.URLError as error:
        reason = error.reason
        report.update(
            error_type=type(error).__name__,
            network_reason_type=type(reason).__name__,
            ssl_library=getattr(reason, "library", None),
            ssl_reason=getattr(reason, "reason", None),
            ssl_verify_code=getattr(reason, "verify_code", None),
            ssl_verify_message=getattr(reason, "verify_message", None),
        )

    except Exception as error:
        report["error_type"] = type(error).__name__

    finally:
        request.remove_header("Authorization")
        report["elapsed_seconds"] = round(time.monotonic() - started, 2)

    return report


try:
    API04_REPORT = api04_run()
except Exception as error:
    API04_REPORT = {
        "status": "LOCAL_SETUP_FAILED",
        "error_type": type(error).__name__,
    }

print(json.dumps(API04_REPORT, ensure_ascii=False, indent=2))

{
  "status": "NEW_GATEWAY_TEST_FAILED",
  "api_request_attempts": 1,
  "automatic_retries": 0,
  "http_status": 502,
  "response_model": null,
  "expected_json_received": false,
  "usage": null,
  "error_type": "HTTPError",
  "project_data_used": false,
  "authorization_scheme": "apikey",
  "env_file_found": true,
  "gateway_url_present": true,
  "gateway_key_present": true,
  "configured_model": "Qwen3-30B-A3B",
  "error_body_bytes": 140,
  "error_body_sha256": "9dc38cea5772e62fb54a88132a65af6fd475bd22addb43d6e5d62ceb38959c4d",
  "error_body_preview": "{\"error\":\"chat completion failed after parallel attempts\",\"status\":502,\"body\":\"Post \\\"https://openrouter.ai/api/v1/chat/completions\\\": EOF\"}",
  "diagnostic_headers": {
    "x-request-id": "4cc8101ff2daa7867b5934e4439d26ca",
    "date": "Tue, 06 Oct 2026 23:49:01 GMT",
    "server": "ArvanCloud"
  },
  "elapsed_seconds": 7.73
}


In [21]:
# API-DIAG-C06 — Test GPT-5.6-Terra; one request, no retries
import json
import time
import ssl
import hashlib
import urllib.request
import urllib.error
from pathlib import Path


def api06_run():
    report = {
        "status": "CONFIG_NOT_READY",
        "configured_model": "GPT-5.6-Terra",
        "api_request_attempts": 0,
        "automatic_retries": 0,
        "http_status": None,
        "expected_json_received": False,
        "response_model": None,
        "usage": None,
        "project_data_used": False,
        "authorization_scheme": "apikey",
    }

    env_path = Path(
        "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
        "RAG-projects/RAG_finance/.env"
    )
    report["env_file_found"] = env_path.is_file()
    if not env_path.is_file():
        report["status"] = "PROJECT_ENV_NOT_FOUND"
        return report

    try:
        from dotenv import dotenv_values
    except ImportError:
        config = {}
        for line in env_path.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[7:].strip()
            name, separator, value = line.partition("=")
            if not separator:
                continue
            value = value.strip()
            if value.startswith(("'", '"')):
                end = value.find(value[0], 1)
                if end < 0:
                    report["status"] = "ENV_QUOTING_INVALID"
                    return report
                value = value[1:end]
            else:
                value = value.split(" #", 1)[0].rstrip()
            config[name.strip()] = value
    else:
        config = dotenv_values(env_path, interpolate=False)

    base = (config.get("ARVAN_GATEWAY_URL") or "").strip().rstrip("/")
    key = (config.get("ARVAN_GATEWAY_API_KEY") or "").strip()

    report.update(
        gateway_url_present=bool(base),
        gateway_key_present=bool(key),
    )
    if base != "https://ai-diag.ragfinance.ir/v1":
        report["status"] = "ENV_URL_IS_NOT_NEW_DIAGNOSTIC_GATEWAY"
        return report
    if not key or "*" in key or key.startswith("<"):
        report["status"] = "KEY_MISSING_OR_MASKED"
        return report

    secrets = {key}
    for name, value in config.items():
        if isinstance(value, str) and value:
            if any(part in name.upper() for part in ("KEY", "TOKEN", "URL")):
                secrets.add(value)

    def safe_text(raw):
        text = raw.decode("utf-8", errors="replace")
        for secret in sorted(secrets, key=len, reverse=True):
            text = text.replace(secret, "[REDACTED]")
        return text[:4000]

    def headers_info(headers):
        return {
            name: headers[name]
            for name in ("x-request-id", "date", "server", "retry-after")
            if headers.get(name) is not None
        }

    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    request = urllib.request.Request(
        base + "/chat/completions",
        data=json.dumps({
            "model": report["configured_model"],
            "messages": [
                {
                    "role": "user",
                    "content": 'Reply only with {"connection":"ok"}.',
                }
            ],
            "max_tokens": 1024,
            "stream": False,
        }).encode("utf-8"),
        headers={
            "Authorization": "apikey " + key,
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )
    opener = urllib.request.build_opener(
        NoRedirect(),
        urllib.request.HTTPSHandler(context=ssl.create_default_context()),
    )

    started = time.monotonic()
    report["api_request_attempts"] = 1
    report["status"] = "SECOND_MODEL_TEST_FAILED"

    try:
        with opener.open(request, timeout=45) as response:
            raw = response.read()
            report["http_status"] = response.status
            report["diagnostic_headers"] = headers_info(response.headers)

        result = json.loads(raw)
        if not isinstance(result, dict):
            report["status"] = "UNEXPECTED_JSON_STRUCTURE"
            return report

        choices = result.get("choices")
        choice = (
            choices[0]
            if isinstance(choices, list) and choices
            and isinstance(choices[0], dict)
            else {}
        )
        message = choice.get("message")
        content = message.get("content") if isinstance(message, dict) else None

        report.update(
            response_model=result.get("model"),
            usage=result.get("usage"),
            finish_reason=choice.get("finish_reason"),
        )
        if isinstance(content, str):
            report["response_content_preview"] = safe_text(
                content.encode("utf-8")
            )
            try:
                report["expected_json_received"] = (
                    json.loads(content) == {"connection": "ok"}
                )
            except ValueError:
                pass

        if "error" in result:
            report["response_error"] = safe_text(
                json.dumps(result["error"], ensure_ascii=False).encode("utf-8")
            )

        report["status"] = (
            "SECOND_MODEL_SMALL_CHAT_PASS"
            if report["http_status"] == 200
            and report["expected_json_received"]
            and report["finish_reason"] == "stop"
            else "SECOND_MODEL_RESPONSE_REQUIRES_INSPECTION"
        )

    except urllib.error.HTTPError as error:
        try:
            body = error.read()
            report.update(
                http_status=error.code,
                error_type=type(error).__name__,
                error_body_bytes=len(body),
                error_body_sha256=hashlib.sha256(body).hexdigest(),
                error_body_preview=safe_text(body),
                diagnostic_headers=headers_info(error.headers),
            )
        finally:
            error.close()

    except urllib.error.URLError as error:
        report.update(
            error_type=type(error).__name__,
            network_reason_type=type(error.reason).__name__,
            ssl_reason=getattr(error.reason, "reason", None),
        )

    except Exception as error:
        report["error_type"] = type(error).__name__

    finally:
        request.remove_header("Authorization")
        report["elapsed_seconds"] = round(time.monotonic() - started, 2)

    return report


try:
    API06_REPORT = api06_run()
except Exception as error:
    API06_REPORT = {
        "status": "LOCAL_SETUP_FAILED",
        "error_type": type(error).__name__,
    }

print(json.dumps(API06_REPORT, ensure_ascii=False, indent=2))

{
  "status": "SECOND_MODEL_SMALL_CHAT_PASS",
  "configured_model": "GPT-5.6-Terra",
  "api_request_attempts": 1,
  "automatic_retries": 0,
  "http_status": 200,
  "expected_json_received": true,
  "response_model": "GPT-5.6-Terra",
  "usage": {
    "prompt_tokens": 15,
    "completion_tokens": 9,
    "total_tokens": 24,
    "prompt_tokens_details": {
      "cached_tokens": 0
    },
    "completion_tokens_details": {
      "reasoning_tokens": 0
    }
  },
  "project_data_used": false,
  "authorization_scheme": "apikey",
  "env_file_found": true,
  "gateway_url_present": true,
  "gateway_key_present": true,
  "diagnostic_headers": {
    "x-request-id": "1bca38c3-f0bd-4948-b80e-1389ee9157a5",
    "date": "Tue, 06 Oct 2026 23:52:02 GMT",
    "server": "ArvanCloud"
  },
  "finish_reason": "stop",
  "response_content_preview": "{\"connection\":\"ok\"}",
  "elapsed_seconds": 3.2
}


In [22]:
# API-DIAG-C07 — DeepSeek-V4-Flash: one request, no retries
import json
import time
import ssl
import hashlib
import urllib.request
import urllib.error
from pathlib import Path


def api07_run():
    report = {
        "status": "CONFIG_NOT_READY",
        "configured_model": "DeepSeek-V4-Flash",
        "api_request_attempts": 0,
        "automatic_retries": 0,
        "http_status": None,
        "expected_json_received": False,
        "response_model": None,
        "usage": None,
        "project_data_used": False,
        "authorization_scheme": "apikey",
    }

    env_path = Path(
        "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
        "RAG-projects/RAG_finance/.env"
    )
    report["env_file_found"] = env_path.is_file()
    if not env_path.is_file():
        report["status"] = "PROJECT_ENV_NOT_FOUND"
        return report

    try:
        from dotenv import dotenv_values
    except ImportError:
        config = {}
        for line in env_path.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[7:].strip()

            name, separator, value = line.partition("=")
            if not separator:
                continue

            value = value.strip()
            if value.startswith(("'", '"')):
                end = value.find(value[0], 1)
                if end < 0:
                    report["status"] = "ENV_QUOTING_INVALID"
                    return report
                value = value[1:end]
            else:
                value = value.split(" #", 1)[0].rstrip()

            config[name.strip()] = value
    else:
        config = dotenv_values(env_path, interpolate=False)

    base = (config.get("ARVAN_GATEWAY_URL") or "").strip().rstrip("/")
    key = (config.get("ARVAN_GATEWAY_API_KEY") or "").strip()

    report.update(
        gateway_url_present=bool(base),
        gateway_key_present=bool(key),
    )
    if base != "https://ai-diag.ragfinance.ir/v1":
        report["status"] = "ENV_URL_IS_NOT_NEW_DIAGNOSTIC_GATEWAY"
        return report
    if not key or "*" in key or key.startswith("<"):
        report["status"] = "KEY_MISSING_OR_MASKED"
        return report

    secrets = {key}
    for name, value in config.items():
        if isinstance(value, str) and value:
            if any(part in name.upper() for part in ("KEY", "TOKEN", "URL")):
                secrets.add(value)

    def safe_text(raw):
        text = raw.decode("utf-8", errors="replace")
        for secret in sorted(secrets, key=len, reverse=True):
            text = text.replace(secret, "[REDACTED]")
        return text[:4000]

    def headers_info(headers):
        return {
            name: headers[name]
            for name in ("x-request-id", "date", "server", "retry-after")
            if headers.get(name) is not None
        }

    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    request = urllib.request.Request(
        base + "/chat/completions",
        data=json.dumps({
            "model": report["configured_model"],
            "messages": [{
                "role": "user",
                "content": 'Reply only with {"connection":"ok"}.',
            }],
            "max_tokens": 1024,
            "stream": False,
        }).encode("utf-8"),
        headers={
            "Authorization": "apikey " + key,
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )
    opener = urllib.request.build_opener(
        NoRedirect(),
        urllib.request.HTTPSHandler(context=ssl.create_default_context()),
    )

    started = time.monotonic()
    report["api_request_attempts"] = 1
    report["status"] = "DEEPSEEK_TEST_FAILED"

    try:
        with opener.open(request, timeout=45) as response:
            raw = response.read()
            report["http_status"] = response.status
            report["diagnostic_headers"] = headers_info(response.headers)

        try:
            result = json.loads(raw)
        except (ValueError, UnicodeDecodeError):
            report["status"] = "NON_JSON_RESPONSE"
            report["response_body_preview"] = safe_text(raw)
            return report

        if not isinstance(result, dict):
            report["status"] = "UNEXPECTED_JSON_STRUCTURE"
            return report

        choices = result.get("choices")
        choice = (
            choices[0]
            if isinstance(choices, list) and choices
            and isinstance(choices[0], dict)
            else {}
        )
        message = choice.get("message")
        content = message.get("content") if isinstance(message, dict) else None

        report.update(
            response_model=result.get("model"),
            usage=result.get("usage"),
            finish_reason=choice.get("finish_reason"),
            model_name_tail_matches=(
                str(result.get("model") or "").split("/")[-1].casefold()
                == report["configured_model"].casefold()
            ),
        )

        if isinstance(content, str):
            report["response_content_preview"] = safe_text(
                content.encode("utf-8")
            )
            try:
                report["expected_json_received"] = (
                    json.loads(content) == {"connection": "ok"}
                )
            except ValueError:
                pass

        if "error" in result:
            report["response_error"] = safe_text(
                json.dumps(result["error"], ensure_ascii=False).encode("utf-8")
            )

        report["status"] = (
            "DEEPSEEK_SMALL_CHAT_PASS"
            if report["http_status"] == 200
            and report["expected_json_received"]
            and report["finish_reason"] == "stop"
            and report["model_name_tail_matches"]
            else "DEEPSEEK_RESPONSE_REQUIRES_INSPECTION"
        )

    except urllib.error.HTTPError as error:
        try:
            body = error.read()
            report.update(
                http_status=error.code,
                error_type=type(error).__name__,
                error_body_bytes=len(body),
                error_body_sha256=hashlib.sha256(body).hexdigest(),
                error_body_preview=safe_text(body),
                diagnostic_headers=headers_info(error.headers),
            )
        finally:
            error.close()

    except urllib.error.URLError as error:
        report.update(
            error_type=type(error).__name__,
            network_reason_type=type(error.reason).__name__,
            ssl_reason=getattr(error.reason, "reason", None),
        )

    except Exception as error:
        report["error_type"] = type(error).__name__

    finally:
        request.remove_header("Authorization")
        report["elapsed_seconds"] = round(time.monotonic() - started, 2)

    return report


try:
    API07_REPORT = api07_run()
except Exception as error:
    API07_REPORT = {
        "status": "LOCAL_SETUP_FAILED",
        "error_type": type(error).__name__,
    }

print(json.dumps(API07_REPORT, ensure_ascii=False, indent=2))

{
  "status": "DEEPSEEK_SMALL_CHAT_PASS",
  "configured_model": "DeepSeek-V4-Flash",
  "api_request_attempts": 1,
  "automatic_retries": 0,
  "http_status": 200,
  "expected_json_received": true,
  "response_model": "DeepSeek-V4-Flash",
  "usage": {
    "completion_tokens": 20,
    "completion_tokens_details": {
      "reasoning_tokens": 14
    },
    "prompt_tokens": 39,
    "prompt_tokens_details": {
      "cached_tokens": 0
    },
    "total_tokens": 59
  },
  "project_data_used": false,
  "authorization_scheme": "apikey",
  "env_file_found": true,
  "gateway_url_present": true,
  "gateway_key_present": true,
  "diagnostic_headers": {
    "x-request-id": "8b5c51dd863b79685d56e0735ebcd049",
    "date": "Tue, 06 Oct 2026 23:58:08 GMT",
    "server": "ArvanCloud"
  },
  "finish_reason": "stop",
  "model_name_tail_matches": true,
  "response_content_preview": "{\"connection\":\"ok\"}",
  "elapsed_seconds": 3.78
}


In [23]:
# API-DIAG-C08 — Compare two judges on saved NB20 case 140
import json
import hashlib
import time
import ssl
import urllib.request
import urllib.error
from pathlib import Path


def api08_run():
    root = Path(
        "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
        "RAG-projects/RAG_finance"
    )
    request_path = root / (
        "artifacts/phase2/answerability_safety_confidence/work/outputs/"
        "nb20_judge_pilot_v1/"
        "b4bb043b41ec030b9ee7475a57a5daf93a9be6dcc4ee3f8a6e53f64b38541d00"
        ".request.json"
    )
    expected_sha = (
        "d9dddc4c1ff9354a3b3d60bdec78b4c8062c3a24bfb493c0b3c42410f9ddadf1"
    )
    report = {
        "status": "LOCAL_PREFLIGHT",
        "position": 140,
        "api_request_attempts": 0,
        "automatic_retries": 0,
        "original_files_changed": False,
        "formal_acceptance": "NOT_ASSESSED",
        "results": [],
    }

    if not request_path.is_file():
        report["status"] = "SAVED_REQUEST_NOT_FOUND"
        return report

    original = request_path.read_bytes()
    if hashlib.sha256(original).hexdigest() != expected_sha:
        report["status"] = "SAVED_REQUEST_HASH_MISMATCH"
        return report

    saved = json.loads(original)
    messages = saved["messages"]
    users = [m for m in messages if m["role"] == "user"]
    assert len(users) == 1, "Unexpected user-message count."
    materials = json.loads(users[0]["content"])
    assert materials["input_profile"] == "QUESTION_EVIDENCE_ONLY"
    schema = materials["schema"]
    payload = materials["review_input"]
    evidence = {e["evidence_id"]: e["text"] for e in payload["evidence"]}
    expected_ids = [e["evidence_id"] for e in payload["evidence"]]

    env_path = root / ".env"
    if not env_path.is_file():
        report["status"] = "PROJECT_ENV_NOT_FOUND"
        return report

    try:
        from dotenv import dotenv_values
    except ImportError:
        config = {}
        for line in env_path.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[7:].strip()
            name, sep, value = line.partition("=")
            if not sep:
                continue
            value = value.strip()
            if value.startswith(("'", '"')):
                end = value.find(value[0], 1)
                if end < 0:
                    report["status"] = "ENV_QUOTING_INVALID"
                    return report
                value = value[1:end]
            else:
                value = value.split(" #", 1)[0].rstrip()
            config[name.strip()] = value
    else:
        config = dotenv_values(env_path, interpolate=False)

    base = (config.get("ARVAN_GATEWAY_URL") or "").strip().rstrip("/")
    key = (config.get("ARVAN_GATEWAY_API_KEY") or "").strip()
    if base != "https://ai-diag.ragfinance.ir/v1":
        report["status"] = "GATEWAY_URL_MISMATCH"
        return report
    if not key or "*" in key or key.startswith("<"):
        report["status"] = "KEY_MISSING_OR_MASKED"
        return report

    secrets = {key}
    for name, value in config.items():
        if isinstance(value, str) and value:
            if any(t in name.upper() for t in ("KEY", "TOKEN", "URL")):
                secrets.add(value)

    def redact(text):
        for secret in sorted(secrets, key=len, reverse=True):
            text = text.replace(secret, "[REDACTED]")
        return text

    def validate(value, rule, path="$"):
        actual = (
            "null" if value is None
            else "boolean" if type(value) is bool
            else "object" if isinstance(value, dict)
            else "array" if isinstance(value, list)
            else "string" if isinstance(value, str)
            else "integer" if type(value) is int
            else "number" if type(value) is float
            else "unknown"
        )
        allowed = rule["type"]
        allowed = allowed if isinstance(allowed, list) else [allowed]
        if actual not in allowed:
            raise ValueError(f"Invalid type: {path}")
        if "enum" in rule and value not in rule["enum"]:
            raise ValueError(f"Invalid enum: {path}")
        if actual == "string":
            if len(value.strip()) < rule.get("minLength", 0):
                raise ValueError(f"Empty field: {path}")
        elif actual == "object":
            if set(value) != set(rule["properties"]):
                raise ValueError(f"Invalid fields: {path}")
            for name, item in value.items():
                validate(item, rule["properties"][name], path + "." + name)
        elif actual == "array":
            for i, item in enumerate(value):
                validate(item, rule["items"], f"{path}[{i}]")

    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    opener = urllib.request.build_opener(
        NoRedirect(),
        urllib.request.HTTPSHandler(context=ssl.create_default_context()),
    )

    # Prevent accidental repeat calls within this notebook kernel.
    seen = globals().setdefault("API08_ATTEMPTED", set())

    for model in ("DeepSeek-V4-Flash", "DeepSeek-V3.2"):
        identity = (expected_sha, base, model)
        row = {
            "model": model,
            "status": "NOT_SENT",
            "http_status": None,
            "schema_pass": False,
            "semantic_acceptance": "NOT_ASSESSED",
        }
        report["results"].append(row)

        if identity in seen:
            row["status"] = "PREVIOUS_ATTEMPT_NO_AUTOMATIC_REPEAT"
            continue

        body = {
            "model": model,
            "messages": messages,
            "max_tokens": saved["max_tokens"],
        }
        request = urllib.request.Request(
            base + "/chat/completions",
            data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
            headers={
                "Authorization": "apikey " + key,
                "Content-Type": "application/json",
                "Accept": "application/json",
            },
            method="POST",
        )

        seen.add(identity)
        report["api_request_attempts"] += 1
        started = time.monotonic()
        print(f"Testing {model} on case 140 ...", flush=True)

        try:
            with opener.open(request, timeout=120) as response:
                raw = response.read()
                row["http_status"] = response.status
                row["request_id"] = response.headers.get("x-request-id")

            response_data = json.loads(raw)
            row["response_model"] = response_data.get("model")
            row["usage"] = response_data.get("usage")
            choice = response_data["choices"][0]
            row["finish_reason"] = choice.get("finish_reason")
            content = choice["message"].get("content")
            row["raw_content"] = (
                redact(content) if isinstance(content, str) else content
            )

            if row["finish_reason"] != "stop":
                row["status"] = "INCOMPLETE_OUTPUT"
                continue

            def reject_constant(value):
                raise ValueError("Non-finite JSON constant.")

            judgment = json.loads(content, parse_constant=reject_constant)
            row["judgment"] = judgment
            validate(judgment, schema)
            row["schema_pass"] = True

            row["evidence_order_pass"] = (
                judgment["all_evidence_reviewed"] is True
                and judgment["evidence_ids_reviewed"] == expected_ids
            )

            quote_checks = []
            for witness in judgment["witnesses"]:
                source = witness["source"]
                reference = witness["reference"]
                quote = witness["quote"]
                text = evidence.get(reference) if source == "EVIDENCE" else None
                start = text.find(quote) if text is not None and quote else -1
                exact_unique = (
                    start >= 0
                    and text.find(quote, start + 1) == -1
                )
                quote_checks.append({
                    "source": source,
                    "reference": reference,
                    "exact_unique": exact_unique,
                })

            row["quote_checks"] = quote_checks
            row["exact_quotes_pass"] = all(
                item["exact_unique"] for item in quote_checks
            )
            row["positive_answer_has_evidence"] = (
                judgment["answerability"]
                not in ("ANSWERABLE", "PARTIALLY_ANSWERABLE")
                or any(w["source"] == "EVIDENCE" for w in judgment["witnesses"])
            )
            row["review_has_reason_and_witness"] = (
                judgment["request_safety"] != "REVIEW_REQUIRED"
                or bool(judgment["safety_reasons"])
                and bool(judgment["witnesses"])
            )
            passed = all(row[name] for name in (
                "schema_pass",
                "evidence_order_pass",
                "exact_quotes_pass",
                "positive_answer_has_evidence",
                "review_has_reason_and_witness",
            ))
            row["status"] = (
                "MECHANICAL_CHECKS_PASS_SEMANTIC_REVIEW_REQUIRED"
                if passed else "MECHANICAL_CHECKS_FAILED"
            )

        except urllib.error.HTTPError as error:
            try:
                raw = error.read()
                row.update(
                    status="HTTP_ERROR",
                    http_status=error.code,
                    error_body=redact(
                        raw.decode("utf-8", errors="replace")
                    )[:4000],
                    request_id=error.headers.get("x-request-id"),
                )
            finally:
                error.close()
        except Exception as error:
            row["status"] = "REQUEST_OR_VALIDATION_FAILED"
            row["error_type"] = type(error).__name__
            if isinstance(error, ValueError):
                row["validation_error"] = redact(str(error))[:500]
        finally:
            request.remove_header("Authorization")
            row["elapsed_seconds"] = round(time.monotonic() - started, 2)

    assert request_path.read_bytes() == original
    report["status"] = "PILOT_COMPLETE_REVIEW_REQUIRED"
    report["project_data_used"] = report["api_request_attempts"] > 0
    return report


try:
    API08_REPORT = api08_run()
except Exception as error:
    API08_REPORT = {
        "status": "LOCAL_OR_PILOT_ERROR",
        "error_type": type(error).__name__,
    }

print(json.dumps(API08_REPORT, ensure_ascii=False, indent=2))

Testing DeepSeek-V4-Flash on case 140 ...
Testing DeepSeek-V3.2 on case 140 ...
{
  "status": "PILOT_COMPLETE_REVIEW_REQUIRED",
  "position": 140,
  "api_request_attempts": 2,
  "automatic_retries": 0,
  "original_files_changed": false,
  "formal_acceptance": "NOT_ASSESSED",
  "results": [
    {
      "model": "DeepSeek-V4-Flash",
      "status": "INCOMPLETE_OUTPUT",
      "http_status": 200,
      "schema_pass": false,
      "semantic_acceptance": "NOT_ASSESSED",
      "request_id": "0de8061beada4245833e9597c8316ddf",
      "response_model": "DeepSeek-V4-Flash",
      "usage": {
        "completion_tokens": 4096,
        "completion_tokens_details": {
          "reasoning_tokens": 4096
        },
        "prompt_tokens": 5623,
        "prompt_tokens_details": {
          "cached_tokens": 0
        },
        "total_tokens": 9719
      },
      "finish_reason": "length",
      "raw_content": "",
      "elapsed_seconds": 27.27
    },
    {
      "model": "DeepSeek-V3.2",
      "status":

In [24]:
# API-DIAG-C09 — Keep V4 result; test V3.2 and GPT-4o-mini
import copy
import json
import hashlib
import time
import ssl
import urllib.request
import urllib.error
from pathlib import Path


def api09_run():
    previous = globals().get("API08_REPORT")
    if not isinstance(previous, dict):
        return {
            "status": "PREVIOUS_REPORT_NOT_IN_KERNEL",
            "api_request_attempts": 0,
        }

    v4_results = [
        row for row in previous.get("results", [])
        if row.get("model") == "DeepSeek-V4-Flash"
    ]
    if len(v4_results) != 1:
        return {
            "status": "PREVIOUS_V4_RESULT_NOT_FOUND",
            "api_request_attempts": 0,
        }

    report = {
        "status": "LOCAL_PREFLIGHT",
        "position": 140,
        "api_request_attempts": 0,
        "automatic_retries": 0,
        "original_files_changed": False,
        "formal_acceptance": "NOT_ASSESSED",
        "results": [copy.deepcopy(v4_results[0])],
    }
    report["results"][0]["execution_action"] = "REUSED_PREVIOUS_REPORT"

    root = Path(
        "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
        "RAG-projects/RAG_finance"
    )
    request_path = root / (
        "artifacts/phase2/answerability_safety_confidence/work/outputs/"
        "nb20_judge_pilot_v1/"
        "b4bb043b41ec030b9ee7475a57a5daf93a9be6dcc4ee3f8a6e53f64b38541d00"
        ".request.json"
    )
    expected_sha = (
        "d9dddc4c1ff9354a3b3d60bdec78b4c8062c3a24bfb493c0b3c42410f9ddadf1"
    )

    if not request_path.is_file():
        report["status"] = "SAVED_REQUEST_NOT_FOUND"
        return report

    original = request_path.read_bytes()
    if hashlib.sha256(original).hexdigest() != expected_sha:
        report["status"] = "SAVED_REQUEST_HASH_MISMATCH"
        return report

    saved = json.loads(original)
    messages = saved["messages"]
    users = [m for m in messages if m["role"] == "user"]
    assert len(users) == 1
    materials = json.loads(users[0]["content"])
    assert materials["input_profile"] == "QUESTION_EVIDENCE_ONLY"

    schema = materials["schema"]
    payload = materials["review_input"]
    evidence = {
        e["evidence_id"]: e["text"] for e in payload["evidence"]
    }
    expected_ids = [e["evidence_id"] for e in payload["evidence"]]

    env_path = root / ".env"
    if not env_path.is_file():
        report["status"] = "PROJECT_ENV_NOT_FOUND"
        return report

    try:
        from dotenv import dotenv_values
    except ImportError:
        config = {}
        for line in env_path.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[7:].strip()
            name, sep, value = line.partition("=")
            if not sep:
                continue
            value = value.strip()
            if value.startswith(("'", '"')):
                end = value.find(value[0], 1)
                if end < 0:
                    report["status"] = "ENV_QUOTING_INVALID"
                    return report
                value = value[1:end]
            else:
                value = value.split(" #", 1)[0].rstrip()
            config[name.strip()] = value
    else:
        config = dotenv_values(env_path, interpolate=False)

    base = (config.get("ARVAN_GATEWAY_URL") or "").strip().rstrip("/")
    key = (config.get("ARVAN_GATEWAY_API_KEY") or "").strip()

    if base != "https://ai-diag.ragfinance.ir/v1":
        report["status"] = "GATEWAY_URL_MISMATCH"
        return report
    if not key or "*" in key or key.startswith("<"):
        report["status"] = "KEY_MISSING_OR_MASKED"
        return report

    secrets = {key}
    for name, value in config.items():
        if isinstance(value, str) and value:
            if any(t in name.upper() for t in ("KEY", "TOKEN", "URL")):
                secrets.add(value)

    def redact(text):
        for secret in sorted(secrets, key=len, reverse=True):
            text = text.replace(secret, "[REDACTED]")
        return text

    def validate(value, rule, path="$"):
        actual = (
            "null" if value is None
            else "boolean" if type(value) is bool
            else "object" if isinstance(value, dict)
            else "array" if isinstance(value, list)
            else "string" if isinstance(value, str)
            else "integer" if type(value) is int
            else "number" if type(value) is float
            else "unknown"
        )
        allowed = rule["type"]
        allowed = allowed if isinstance(allowed, list) else [allowed]

        if actual not in allowed:
            raise ValueError(f"Invalid type: {path}")
        if "enum" in rule and value not in rule["enum"]:
            raise ValueError(f"Invalid enum: {path}")

        if actual == "string":
            if len(value.strip()) < rule.get("minLength", 0):
                raise ValueError(f"Empty field: {path}")
        elif actual == "object":
            if set(value) != set(rule["properties"]):
                raise ValueError(f"Invalid fields: {path}")
            for name, item in value.items():
                validate(item, rule["properties"][name], path + "." + name)
        elif actual == "array":
            for i, item in enumerate(value):
                validate(item, rule["items"], f"{path}[{i}]")

    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    opener = urllib.request.build_opener(
        NoRedirect(),
        urllib.request.HTTPSHandler(context=ssl.create_default_context()),
    )

    # This cell has its own attempt guard:
    # V3.2 is retried once because it has now been added to the gateway.
    seen = globals().setdefault("API09_ATTEMPTED", set())

    for model in ("DeepSeek-V3.2", "GPT-4o-mini"):
        identity = (expected_sha, base, model)
        row = {
            "model": model,
            "status": "NOT_SENT",
            "execution_action": "NEW_TEST",
            "http_status": None,
            "schema_pass": False,
            "semantic_acceptance": "NOT_ASSESSED",
        }
        report["results"].append(row)

        if identity in seen:
            row["status"] = "PREVIOUS_ATTEMPT_NO_AUTOMATIC_REPEAT"
            continue

        request = urllib.request.Request(
            base + "/chat/completions",
            data=json.dumps({
                "model": model,
                "messages": messages,
                "max_tokens": saved["max_tokens"],
            }, ensure_ascii=False).encode("utf-8"),
            headers={
                "Authorization": "apikey " + key,
                "Content-Type": "application/json",
                "Accept": "application/json",
            },
            method="POST",
        )

        seen.add(identity)
        report["api_request_attempts"] += 1
        started = time.monotonic()
        print(f"Testing {model} on case 140 ...", flush=True)

        try:
            with opener.open(request, timeout=120) as response:
                raw = response.read()
                row["http_status"] = response.status
                row["request_id"] = response.headers.get("x-request-id")

            response_data = json.loads(raw)
            row["response_model"] = response_data.get("model")
            row["usage"] = response_data.get("usage")

            choice = response_data["choices"][0]
            row["finish_reason"] = choice.get("finish_reason")
            content = choice["message"].get("content")
            row["raw_content"] = (
                redact(content) if isinstance(content, str) else content
            )

            if row["finish_reason"] != "stop":
                row["status"] = "INCOMPLETE_OUTPUT"
                continue

            def reject_constant(value):
                raise ValueError("Non-finite JSON constant.")

            judgment = json.loads(content, parse_constant=reject_constant)
            row["judgment"] = judgment
            validate(judgment, schema)
            row["schema_pass"] = True

            row["evidence_order_pass"] = (
                judgment["all_evidence_reviewed"] is True
                and judgment["evidence_ids_reviewed"] == expected_ids
            )

            quote_checks = []
            for witness in judgment["witnesses"]:
                source = witness["source"]
                reference = witness["reference"]
                quote = witness["quote"]
                text = evidence.get(reference) if source == "EVIDENCE" else None
                start = text.find(quote) if text is not None and quote else -1
                exact_unique = (
                    start >= 0 and text.find(quote, start + 1) == -1
                )
                quote_checks.append({
                    "source": source,
                    "reference": reference,
                    "exact_unique": exact_unique,
                })

            row["quote_checks"] = quote_checks
            row["exact_quotes_pass"] = all(
                item["exact_unique"] for item in quote_checks
            )
            row["positive_answer_has_evidence"] = (
                judgment["answerability"]
                not in ("ANSWERABLE", "PARTIALLY_ANSWERABLE")
                or any(w["source"] == "EVIDENCE" for w in judgment["witnesses"])
            )
            row["review_has_reason_and_witness"] = (
                judgment["request_safety"] != "REVIEW_REQUIRED"
                or (
                    bool(judgment["safety_reasons"])
                    and bool(judgment["witnesses"])
                )
            )

            passed = all(row[name] for name in (
                "schema_pass",
                "evidence_order_pass",
                "exact_quotes_pass",
                "positive_answer_has_evidence",
                "review_has_reason_and_witness",
            ))
            row["status"] = (
                "MECHANICAL_CHECKS_PASS_SEMANTIC_REVIEW_REQUIRED"
                if passed else "MECHANICAL_CHECKS_FAILED"
            )

        except urllib.error.HTTPError as error:
            try:
                raw = error.read()
                row.update(
                    status="HTTP_ERROR",
                    http_status=error.code,
                    error_body=redact(
                        raw.decode("utf-8", errors="replace")
                    )[:4000],
                    request_id=error.headers.get("x-request-id"),
                )
            finally:
                error.close()
        except Exception as error:
            row["status"] = "REQUEST_OR_VALIDATION_FAILED"
            row["error_type"] = type(error).__name__
            if isinstance(error, ValueError):
                row["validation_error"] = redact(str(error))[:500]
        finally:
            request.remove_header("Authorization")
            row["elapsed_seconds"] = round(time.monotonic() - started, 2)

    assert request_path.read_bytes() == original
    report["status"] = "THREE_MODEL_PILOT_REVIEW_REQUIRED"
    report["project_data_used"] = report["api_request_attempts"] > 0
    return report


try:
    API09_REPORT = api09_run()
except Exception as error:
    API09_REPORT = {
        "status": "LOCAL_OR_PILOT_ERROR",
        "error_type": type(error).__name__,
    }

print(json.dumps(API09_REPORT, ensure_ascii=False, indent=2))

Testing DeepSeek-V3.2 on case 140 ...
Testing GPT-4o-mini on case 140 ...
{
  "status": "THREE_MODEL_PILOT_REVIEW_REQUIRED",
  "position": 140,
  "api_request_attempts": 2,
  "automatic_retries": 0,
  "original_files_changed": false,
  "formal_acceptance": "NOT_ASSESSED",
  "results": [
    {
      "model": "DeepSeek-V4-Flash",
      "status": "INCOMPLETE_OUTPUT",
      "http_status": 200,
      "schema_pass": false,
      "semantic_acceptance": "NOT_ASSESSED",
      "request_id": "0de8061beada4245833e9597c8316ddf",
      "response_model": "DeepSeek-V4-Flash",
      "usage": {
        "completion_tokens": 4096,
        "completion_tokens_details": {
          "reasoning_tokens": 4096
        },
        "prompt_tokens": 5623,
        "prompt_tokens_details": {
          "cached_tokens": 0
        },
        "total_tokens": 9719
      },
      "finish_reason": "length",
      "raw_content": "",
      "elapsed_seconds": 27.27,
      "execution_action": "REUSED_PREVIOUS_REPORT"
    },
    

In [25]:
# API-DIAG-C10 — Test GPT-5-Mini on saved NB20 case 140
import copy
import json
import hashlib
import time
import ssl
import urllib.request
import urllib.error
from pathlib import Path


def api10_run():
    previous = globals().get("API09_REPORT")
    report = {
        "status": "LOCAL_PREFLIGHT",
        "position": 140,
        "api_request_attempts": 0,
        "automatic_retries": 0,
        "original_files_changed": False,
        "formal_acceptance": "NOT_ASSESSED",
        "results": [],
    }

    if isinstance(previous, dict):
        report["results"] = copy.deepcopy(previous.get("results", []))
        for row in report["results"]:
            row["execution_action"] = "REUSED_PREVIOUS_REPORT"

    root = Path(
        "/Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/"
        "RAG-projects/RAG_finance"
    )
    request_path = root / (
        "artifacts/phase2/answerability_safety_confidence/work/outputs/"
        "nb20_judge_pilot_v1/"
        "b4bb043b41ec030b9ee7475a57a5daf93a9be6dcc4ee3f8a6e53f64b38541d00"
        ".request.json"
    )
    expected_sha = (
        "d9dddc4c1ff9354a3b3d60bdec78b4c8062c3a24bfb493c0b3c42410f9ddadf1"
    )

    if not request_path.is_file():
        report["status"] = "SAVED_REQUEST_NOT_FOUND"
        return report

    original = request_path.read_bytes()
    if hashlib.sha256(original).hexdigest() != expected_sha:
        report["status"] = "SAVED_REQUEST_HASH_MISMATCH"
        return report

    saved = json.loads(original)
    messages = saved["messages"]
    users = [m for m in messages if m["role"] == "user"]
    assert len(users) == 1

    materials = json.loads(users[0]["content"])
    assert materials["input_profile"] == "QUESTION_EVIDENCE_ONLY"
    schema = materials["schema"]
    payload = materials["review_input"]
    evidence = {
        e["evidence_id"]: e["text"] for e in payload["evidence"]
    }
    expected_ids = [e["evidence_id"] for e in payload["evidence"]]

    env_path = root / ".env"
    if not env_path.is_file():
        report["status"] = "PROJECT_ENV_NOT_FOUND"
        return report

    try:
        from dotenv import dotenv_values
    except ImportError:
        config = {}
        for line in env_path.read_text(encoding="utf-8-sig").splitlines():
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[7:].strip()
            name, sep, value = line.partition("=")
            if not sep:
                continue
            value = value.strip()
            if value.startswith(("'", '"')):
                end = value.find(value[0], 1)
                if end < 0:
                    report["status"] = "ENV_QUOTING_INVALID"
                    return report
                value = value[1:end]
            else:
                value = value.split(" #", 1)[0].rstrip()
            config[name.strip()] = value
    else:
        config = dotenv_values(env_path, interpolate=False)

    base = (config.get("ARVAN_GATEWAY_URL") or "").strip().rstrip("/")
    key = (config.get("ARVAN_GATEWAY_API_KEY") or "").strip()
    model = "GPT-5-Mini"

    if base != "https://ai-diag.ragfinance.ir/v1":
        report["status"] = "GATEWAY_URL_MISMATCH"
        return report
    if not key or "*" in key or key.startswith("<"):
        report["status"] = "KEY_MISSING_OR_MASKED"
        return report

    secrets = {key}
    for name, value in config.items():
        if isinstance(value, str) and value:
            if any(t in name.upper() for t in ("KEY", "TOKEN", "URL")):
                secrets.add(value)

    def redact(text):
        for secret in sorted(secrets, key=len, reverse=True):
            text = text.replace(secret, "[REDACTED]")
        return text

    def schema_errors(value, rule, path="$"):
        errors = []
        actual = (
            "null" if value is None
            else "boolean" if type(value) is bool
            else "object" if isinstance(value, dict)
            else "array" if isinstance(value, list)
            else "string" if isinstance(value, str)
            else "integer" if type(value) is int
            else "number" if type(value) is float
            else "unknown"
        )
        allowed = rule["type"]
        allowed = allowed if isinstance(allowed, list) else [allowed]

        if actual not in allowed:
            return [f"Invalid type: {path}"]
        if "enum" in rule and value not in rule["enum"]:
            errors.append(f"Invalid enum: {path}")

        if actual == "string":
            if len(value.strip()) < rule.get("minLength", 0):
                errors.append(f"Empty field: {path}")
        elif actual == "object":
            properties = rule["properties"]
            for name in rule.get("required", []):
                if name not in value:
                    errors.append(f"Missing field: {path}.{name}")
            for name, item in value.items():
                if name not in properties:
                    if rule.get("additionalProperties") is False:
                        errors.append(f"Unexpected field: {path}.{name}")
                else:
                    errors.extend(
                        schema_errors(item, properties[name], path + "." + name)
                    )
        elif actual == "array":
            for i, item in enumerate(value):
                errors.extend(
                    schema_errors(item, rule["items"], f"{path}[{i}]")
                )
        return errors

    row = {
        "model": model,
        "status": "NOT_SENT",
        "execution_action": "NEW_TEST",
        "http_status": None,
        "schema_pass": False,
        "semantic_acceptance": "NOT_ASSESSED",
    }
    report["results"].append(row)

    seen = globals().setdefault("API10_ATTEMPTED", set())
    identity = (expected_sha, base, model)
    if identity in seen:
        row["status"] = "PREVIOUS_ATTEMPT_NO_AUTOMATIC_REPEAT"
        report["status"] = "NO_NEW_REQUEST"
        return report

    class NoRedirect(urllib.request.HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            return None

    # Keep the same gateway request format and output cap as previous tests.
    body = {
        "model": model,
        "messages": messages,
        "max_tokens": saved["max_tokens"],
    }
    request = urllib.request.Request(
        base + "/chat/completions",
        data=json.dumps(body, ensure_ascii=False).encode("utf-8"),
        headers={
            "Authorization": "apikey " + key,
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )
    opener = urllib.request.build_opener(
        NoRedirect(),
        urllib.request.HTTPSHandler(context=ssl.create_default_context()),
    )

    seen.add(identity)
    report["api_request_attempts"] = 1
    started = time.monotonic()
    print(f"Testing {model} on case 140 ...", flush=True)

    try:
        with opener.open(request, timeout=120) as response:
            raw = response.read()
            row["http_status"] = response.status
            row["request_id"] = response.headers.get("x-request-id")

        response_data = json.loads(raw)
        row["response_model"] = response_data.get("model")
        row["usage"] = response_data.get("usage")
        choice = response_data["choices"][0]
        row["finish_reason"] = choice.get("finish_reason")
        content = choice["message"].get("content")
        row["raw_content"] = (
            redact(content) if isinstance(content, str) else content
        )

        if row["finish_reason"] != "stop":
            row["status"] = "INCOMPLETE_OUTPUT"
        else:
            def reject_constant(value):
                raise ValueError("Non-finite JSON constant.")

            judgment = json.loads(content, parse_constant=reject_constant)
            row["judgment"] = judgment
            errors = schema_errors(judgment, schema)
            row["schema_errors"] = errors
            row["schema_pass"] = not errors

            if errors:
                row["status"] = "SCHEMA_FAILED"
            else:
                row["evidence_order_pass"] = (
                    judgment["all_evidence_reviewed"] is True
                    and judgment["evidence_ids_reviewed"] == expected_ids
                )

                quote_checks = []
                for witness in judgment["witnesses"]:
                    source = witness["source"]
                    reference = witness["reference"]
                    quote = witness["quote"]
                    text = (
                        evidence.get(reference)
                        if source == "EVIDENCE" else None
                    )
                    start = (
                        text.find(quote)
                        if text is not None and quote else -1
                    )
                    quote_checks.append({
                        "source": source,
                        "reference": reference,
                        "reference_valid": text is not None,
                        "exact_unique": (
                            start >= 0
                            and text.find(quote, start + 1) == -1
                        ),
                    })

                row["quote_checks"] = quote_checks
                row["exact_quotes_pass"] = all(
                    item["exact_unique"] for item in quote_checks
                )
                row["positive_answer_has_evidence"] = (
                    judgment["answerability"]
                    not in ("ANSWERABLE", "PARTIALLY_ANSWERABLE")
                    or any(
                        w["source"] == "EVIDENCE"
                        for w in judgment["witnesses"]
                    )
                )
                row["review_has_reason_and_witness"] = (
                    judgment["request_safety"] != "REVIEW_REQUIRED"
                    or (
                        bool(judgment["safety_reasons"])
                        and bool(judgment["witnesses"])
                    )
                )

                passed = all(row[name] for name in (
                    "schema_pass",
                    "evidence_order_pass",
                    "exact_quotes_pass",
                    "positive_answer_has_evidence",
                    "review_has_reason_and_witness",
                ))
                row["status"] = (
                    "MECHANICAL_CHECKS_PASS_SEMANTIC_REVIEW_REQUIRED"
                    if passed else "MECHANICAL_CHECKS_FAILED"
                )

    except urllib.error.HTTPError as error:
        try:
            raw = error.read()
            row.update(
                status="HTTP_ERROR",
                http_status=error.code,
                error_body=redact(
                    raw.decode("utf-8", errors="replace")
                )[:4000],
                request_id=error.headers.get("x-request-id"),
            )
        finally:
            error.close()
    except Exception as error:
        row["status"] = "REQUEST_OR_VALIDATION_FAILED"
        row["error_type"] = type(error).__name__
        if isinstance(error, ValueError):
            row["validation_error"] = redact(str(error))[:500]
    finally:
        request.remove_header("Authorization")
        row["elapsed_seconds"] = round(time.monotonic() - started, 2)

    assert request_path.read_bytes() == original
    report["status"] = "PILOT_REVIEW_REQUIRED"
    report["project_data_used"] = True
    return report


try:
    API10_REPORT = api10_run()
except Exception as error:
    API10_REPORT = {
        "status": "LOCAL_OR_PILOT_ERROR",
        "error_type": type(error).__name__,
    }

print(json.dumps(API10_REPORT, ensure_ascii=False, indent=2))

Testing GPT-5-Mini on case 140 ...
{
  "status": "PILOT_REVIEW_REQUIRED",
  "position": 140,
  "api_request_attempts": 1,
  "automatic_retries": 0,
  "original_files_changed": false,
  "formal_acceptance": "NOT_ASSESSED",
  "results": [
    {
      "model": "DeepSeek-V4-Flash",
      "status": "INCOMPLETE_OUTPUT",
      "http_status": 200,
      "schema_pass": false,
      "semantic_acceptance": "NOT_ASSESSED",
      "request_id": "0de8061beada4245833e9597c8316ddf",
      "response_model": "DeepSeek-V4-Flash",
      "usage": {
        "completion_tokens": 4096,
        "completion_tokens_details": {
          "reasoning_tokens": 4096
        },
        "prompt_tokens": 5623,
        "prompt_tokens_details": {
          "cached_tokens": 0
        },
        "total_tokens": 9719
      },
      "finish_reason": "length",
      "raw_content": "",
      "elapsed_seconds": 27.27,
      "execution_action": "REUSED_PREVIOUS_REPORT"
    },
    {
      "model": "DeepSeek-V3.2",
      "status": "